# Cardiomegaly Classifier — Training Notebook

This notebook is intentionally minimal. All logic lives in `src/`.

| File | Responsibility |
|------|----------------|
| `src/config.py` | `Config` dataclass + global `CFG` |
| `src/utils.py` | `set_seed`, `free_device_cache` |
| `src/data.py` | `load_labels`, `split_dataframe`, `estimate_gray_mean_std` |
| `src/dataset.py` | `CardiomegalyDataset`, `SubmissionDataset` |
| `src/transforms.py` | `make_transforms`, `make_tta_transforms` |
| `src/model.py` | `CardiomegalyModel`, `build_model`, freeze helpers |
| `src/train.py` | `train_model`, `run_epoch_tta`, `optuna_search`, `save_results` |

## 1 · Setup

In [ ]:
import sys, os
# Make sure the project root (new_structure/) is on the path
sys.path.insert(0, os.path.abspath(".."))

from src.config import CFG
from src.utils import set_seed

set_seed(CFG.seed)
print("Device :", CFG.device)
print("img_size:", CFG.img_size)
print("Results :", CFG.output_dir)

## 2 · Data Loading & Splitting

In [ ]:
from src.data import load_labels, split_dataframe, estimate_gray_mean_std

full_df                   = load_labels(CFG.csv_path, CFG.image_dir)
train_df, val_df, test_df = split_dataframe(full_df)
train_mean, train_std     = estimate_gray_mean_std(train_df, CFG.stats_sample_size)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print(f"Grayscale mean: {train_mean:.6f}  std: {train_std:.6f}")

## 3 · Datasets & DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import CardiomegalyDataset
from src.transforms import make_transforms

train_tf, eval_tf = make_transforms(train_mean, train_std)

train_ds = CardiomegalyDataset(train_df, transform=train_tf)
val_ds   = CardiomegalyDataset(val_df,   transform=eval_tf)
test_ds  = CardiomegalyDataset(test_df,  transform=eval_tf)

_kw = dict(batch_size=CFG.batch_size, num_workers=CFG.num_workers, pin_memory=False)
train_loader = DataLoader(train_ds, shuffle=True,  **_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **_kw)

print(f"Batches — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")

## 4 · Hyperparameter Search (optional)

Skip this section to train with the default `CFG` values.
Run it to let Optuna find better hyperparameters automatically.

In [ ]:
# from src.train import optuna_search, apply_best_params

# N_TRIALS = 20  # increase for more thorough search (~1-2 min per trial on MPS)

# study = optuna_search(train_ds, val_ds, train_df, n_trials=N_TRIALS)
# train_loader, val_loader, test_loader, BEST_N_BLOCKS = apply_best_params(
#     study, train_ds, val_ds, test_ds
# )

## 5 · Train Model

In [ ]:
from src.model import build_model
from src.train import train_model

n_blocks = 7
# BEST_N_BLOCKS if "BEST_N_BLOCKS" in vars() else 7

model   = build_model(CFG.dropout).to(CFG.device)
model, history = train_model(
    model, train_loader, val_loader, train_df, n_blocks=n_blocks
)

## 6 · TTA Evaluation & Threshold Selection

In [ ]:
from src.train import run_epoch_tta, find_best_threshold, compute_basic_metrics
from src.transforms import make_tta_transforms

tta_transforms = make_tta_transforms(train_mean, train_std)

val_out  = run_epoch_tta(model, val_df,  tta_transforms)
test_out = run_epoch_tta(model, test_df, tta_transforms)

print(f"Val  AUC (TTA): {val_out['auc']:.4f}")
print(f"Test AUC (TTA): {test_out['auc']:.4f}")

best_threshold, _ = find_best_threshold(val_out["y_true"], val_out["y_prob"], mode="youden")
print(f"\nSelected threshold (Youden): {best_threshold:.4f}")

for name, out in [("Validation", val_out), ("Test", test_out)]:
    m = compute_basic_metrics(out["y_true"], out["y_prob"], best_threshold)
    print(f"\n{name}: AUC={m['auc']:.4f}  Sens={m['sensitivity']:.4f}  Spec={m['specificity']:.4f}")

## 7 · ROC Curves

In [ ]:
import os
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, out, title in [(axes[0], val_out, "Validation"), (axes[1], test_out, "Test")]:
    fpr, tpr, _ = roc_curve(out["y_true"], out["y_prob"])
    ax.plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{title} ROC Curve")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG.output_dir, "roc_curves.png"), dpi=150)
plt.show()

## 8 · Save Results

In [ ]:
from src.train import save_results

# Set a descriptive name — shown in results_log.csv for easy comparison
MODEL_NAME = "efficientnet_b3_tabular"

save_results(
    model=model,
    history=history,
    val_out=val_out,
    test_out=test_out,
    best_threshold=best_threshold,
    output_dir=CFG.output_dir,
    model_name=MODEL_NAME,
    n_blocks=n_blocks,
    config=CFG,
)

## 9 · Submission (optional)

Only needed if unlabelled test images exist at `CFG.submission_test_dir`.

In [ ]:
import os, pandas as pd
from src.train import predict_submission_tta

if os.path.isdir(CFG.submission_test_dir):
    submission_out = predict_submission_tta(model, CFG.submission_test_dir, tta_transforms)
    sub_df = pd.DataFrame({
        "image_file": submission_out["names"],
        "prob":       submission_out["y_prob"],
    })
    sub_df["pred"] = (sub_df["prob"] >= best_threshold).astype(int)
    sub_path = os.path.join(CFG.output_dir, "daily_submission.csv")
    sub_df.to_csv(sub_path, index=False)
    print(f"Submission saved → {sub_path}  ({len(sub_df)} rows)")
    display(sub_df.head())
else:
    print(f"No submission directory found at: {CFG.submission_test_dir}")